# Two-bit fermionic Fourier transform

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0. [Locked source notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/qft/two_bit_ffft.ipynb).

In [ ]:
from IPython.display import SVG, display
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.field_addition_catalog import field_signature_svg

## Two-bit Fourier gate

$F(k,n)$ mixes the two single-occupation states with relative phase $e^{2\pi i k/n}$ and gives the doubly occupied state a minus sign times that phase. Here $k$ selects the Fourier phase, $n$ is nonzero, and epsilon controls the resource declaration for the leading rotation. The transform is the building block in Eq. 32 and Fig. 8 of [Improved Fault-Tolerant Quantum Simulation of Condensed-Phase Correlated Electrons via Trotterization](https://arxiv.org/abs/1902.10673).

In [ ]:
from flagquantum.algorithms.primitives.fermionic_fourier import (
    source_two_bit_fermionic_fourier_signature, source_two_bit_fermionic_fourier_cost,
    two_bit_fermionic_fourier_ir,
)
from flagquantum.simulation.fermionic_fourier import two_bit_fermionic_fourier_matrix
from examples.algorithms.two_bit_fermionic_fourier_catalog import (
    TWO_BIT_FOURIER_EXAMPLE, two_bit_fermionic_fourier_graph,
)

### Example instance

In [ ]:
k, n = TWO_BIT_FOURIER_EXAMPLE
two_bit_fermionic_fourier_matrix(k, n)

#### Graphical signature

Both occupation bits are through ports.

In [ ]:
display(SVG(field_signature_svg(((
    f"F({k}, {n})", source_two_bit_fermionic_fourier_signature(), {},
),))))

### Call graph

The locked resource declaration counts T and its adjoint together. Its rotation precision remains part of the rotation key.

In [ ]:
graph = two_bit_fermionic_fourier_graph(k, n)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
print("Declared leaves:", graph.leaf_counts)
print("Source model:", source_two_bit_fermionic_fourier_cost(k, n).gates)

### Execute through append

The physical forward circuit has the global phase $e^{-i\pi k/n}$ relative to the analytical tensor. The native inverse reverses the gates and conjugates this phase. The locked decomposition emits the forward circuit even when its adjoint flag is set.

In [ ]:
from cmath import exp
from collections import Counter
from math import pi
import torch
from flagquantum.circuit import Circuit
from examples.algorithms.binary_field_arithmetic import FieldOperation

state = torch.tensor([1+1j, 2-1j, -3j, 4+2j], dtype=torch.complex128)
state /= state.norm()
forward = two_bit_fermionic_fourier_ir(0, 1, k, n)
inverse = two_bit_fermionic_fourier_ir(0, 1, k, n, adjoint=True)
print("Emitted primitives:", Counter(gate.name for gate in forward))
circuit = Circuit(2, inputs=state, dtype=torch.complex128)
circuit.append(FieldOperation(2, forward))
phase = exp(-1j*pi*k/n)
tensor = torch.tensor(two_bit_fermionic_fourier_matrix(k, n))
torch.testing.assert_close(circuit.state().flatten(), phase * (tensor @ state))
circuit.append(FieldOperation(2, inverse))
torch.testing.assert_close(circuit.state().flatten(), state)